In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094903.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095045.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094723.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094736.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094812.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094741.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094755.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094947.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094825.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094713.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094907.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094835.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094952.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095049.jpg
/kaggle/input/datasets/ashok8yadav/light-data/IM

In [2]:
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

class CustomImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform

        self.files = sorted([
            f for f in os.listdir(root)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        path = os.path.join(self.root, self.files[idx])
        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image


# SAME VARIABLE NAME (so your old code works)
train_dataset = CustomImageDataset(
    root="/kaggle/input/datasets/ashok8yadav/light-data/",
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

print("Images:", len(train_dataset))

Images: 36


In [3]:
#auxilary code 
import torch
import torch.nn as nn

class AutoEncoder(nn.Module):
    def __init__(self):
        super().__init__()

        # ---------------- ENCODER ----------------
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.Conv2d(64, 128, 3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU()
        )

        # ---------------- DECODER ----------------
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),

            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),

            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.decoder(x)
        return x


# ---------------- DEVICE SETUP ----------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoEncoder().to(device)

print("✅ Improved AutoEncoder Model Ready on:", device)

✅ Improved AutoEncoder Model Ready on: cpu


In [4]:
import torch
import torch.nn as nn

criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 20

model.train()

for epoch in range(epochs):

    total_loss = 0

    for images in train_loader:

        images = images.to(device)

        outputs = model(images)
        loss = criterion(outputs, images)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.6f}")

Epoch [1/20] Loss: 0.072605
Epoch [2/20] Loss: 0.047923
Epoch [3/20] Loss: 0.036061
Epoch [4/20] Loss: 0.030056
Epoch [5/20] Loss: 0.024592
Epoch [6/20] Loss: 0.018424
Epoch [7/20] Loss: 0.015610
Epoch [8/20] Loss: 0.013689
Epoch [9/20] Loss: 0.011644
Epoch [10/20] Loss: 0.011142
Epoch [11/20] Loss: 0.009496
Epoch [12/20] Loss: 0.008550
Epoch [13/20] Loss: 0.008171
Epoch [14/20] Loss: 0.008809
Epoch [15/20] Loss: 0.007580
Epoch [16/20] Loss: 0.007433
Epoch [17/20] Loss: 0.006652
Epoch [18/20] Loss: 0.006685
Epoch [19/20] Loss: 0.006089
Epoch [20/20] Loss: 0.005993


In [5]:
torch.save(model.state_dict(), "light.pth")

In [6]:
import torch
from PIL import Image
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),  # adjust if needed
    transforms.ToTensor()
])

In [7]:
import os

def check_images(model, image_paths, threshold=0.01, device="cuda"):
    model.eval()

    results = []

    with torch.no_grad():
        for path in image_paths:

            img = Image.open(path).convert("RGB")
            img = transform(img).unsqueeze(0).to(device)

            output = model(img)

            loss = torch.mean((output - img) ** 2).item()

            label = "GOOD" if loss < threshold else "NOT GOOD"

            results.append((path, loss, label))

            print(f"{os.path.basename(path)} -> Loss: {loss:.6f} => {label}")

    return results

In [8]:
image_paths = [
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094903.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095045.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094723.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094736.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094812.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094741.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094755.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094947.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094825.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094713.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094907.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094835.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094952.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095049.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094844.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095109.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094855.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094848.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094934.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094938.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094839.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094830.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095040.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094912.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095011.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095104.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094807.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094701.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094748.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095056.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094818.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094957.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095001.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094859.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094728.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095030.jpg'
]

In [9]:
results = check_images(model, good_images, threshold=0.01, device=device)

NameError: name 'good_images' is not defined

In [ ]:
#path core training 

In [ ]:
normal_images = [
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094903.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095045.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094723.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094736.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094812.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094741.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094755.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094947.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094825.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094713.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094907.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094835.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094952.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095049.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094844.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095109.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094855.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094848.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094934.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094938.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094839.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094830.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095040.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094912.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095011.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095104.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094807.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094701.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094748.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095056.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094818.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094957.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095001.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094859.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094728.jpg',
    '/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602095030.jpg'
]

In [ ]:
import torch
import torchvision.models as models

device = "cuda" if torch.cuda.is_available() else "cpu"

backbone = models.resnet18(pretrained=True)
backbone = torch.nn.Sequential(*list(backbone.children())[:-2])
backbone = backbone.to(device)
backbone.eval()

In [ ]:
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
def extract_features(path):
    img = Image.open(path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = backbone(img)

    return feat.squeeze().cpu().flatten()

In [ ]:
memory_bank = []

for path in normal_images:
    feat = extract_features(path)
    memory_bank.append(feat)

memory_bank = torch.stack(memory_bank)
print("Memory bank shape:", memory_bank.shape)

In [ ]:
import torch.nn.functional as F

def anomaly_score(image_path):
    feat = extract_features(image_path)

    # L2 distance to memory bank
    dist = torch.cdist(feat.unsqueeze(0), memory_bank)

    score = dist.min().item()  # closest normal image distance

    return score

In [ ]:
def predict(image_path, threshold=5.0):
    score = anomaly_score(image_path)

    label = "BAD (Defect)" if score > threshold else "GOOD"

    print(f"{image_path} -> Score: {score:.4f} => {label}")

    return score, label

In [ ]:
test_image = "/kaggle/input/datasets/ashok8yadav/light-data/IMG20260602094903.jpg"

predict(test_image)

In [ ]:
def test_all_images(image_paths, threshold=5.0):
    
    results = []
    
    good_count = 0
    bad_count = 0

    for path in image_paths:
        score = anomaly_score(path)
        
        if score > threshold:
            label = "BAD"
            bad_count += 1
        else:
            label = "GOOD"
            good_count += 1

        print(f"{path} -> {score:.4f} => {label}")
        
        results.append((path, score, label))

    print("\n========== SUMMARY ==========")
    print(f"TOTAL IMAGES: {len(image_paths)}")
    print(f"GOOD: {good_count}")
    print(f"BAD: {bad_count}")

    return results

In [ ]:
results = test_all_images(notgood_images, threshold=5.0)

In [ ]:
# version 2.00

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
from torchvision import transforms
from PIL import Image
import numpy as np

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
backbone = models.resnet18(pretrained=True)
backbone = torch.nn.Sequential(*list(backbone.children())[:-2])  # keep feature maps
backbone = backbone.to(device)
backbone.eval()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [ ]:
def extract_features(path):
    img = Image.open(path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        feat = backbone(img)

    return feat.squeeze().cpu().flatten()

In [ ]:
memory_bank = []

for path in normal_images:
    feat = extract_features(path)
    memory_bank.append(feat)

memory_bank = torch.stack(memory_bank)

print("Memory bank shape:", memory_bank.shape)

In [ ]:
def anomaly_score(image_path):
    feat = extract_features(image_path)

    dist = torch.cdist(feat.unsqueeze(0), memory_bank)

    score = dist.min().item()

    return score

In [ ]:
scores = [anomaly_score(p) for p in normal_images]

threshold = np.mean(scores) + np.std(scores)

print("Calculated threshold:", threshold)

In [ ]:
def predict(image_path):
    score = anomaly_score(image_path)

    if score > threshold:
        label = "NOT GOOD"
    else:
        label = "GOOD"

    print(f"{image_path}")
    print(f"Score: {score:.4f} -> {label}")

    return score, label

In [ ]:
good_images = [
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121745.jpg",
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121720.jpg",
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121715.jpg",
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121638.jpg",
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121728.jpg",
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121646.jpg",
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121752.jpg",
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121651.jpg",
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121710.jpg",
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121603.jpg",
"/kaggle/input/datasets/ashok8yadav/good-light/IMG20260602121611.jpg",
]

# not-good images
notgood_images = [
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602122315.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135430.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135342.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602122348.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135353.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135457.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135510.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602122334.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135450.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602122224.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602122416.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135337.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135455.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135524.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135414.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135453.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135446.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135402.jpg",
"/kaggle/input/datasets/ashok8yadav/not-good/IMG20260602135348.jpg",
]

In [ ]:
def test_all(image_list):
    good = 0
    bad = 0

    for img in image_list:
        score, label = predict(img)

        if label == "GOOD":
            good += 1
        else:
            bad += 1

    print("\n===== FINAL RESULT =====")
    print("TOTAL:", len(image_list))
    print("GOOD:", good)
    print("NOT GOOD:", bad)

test_all(notgood_images)